# Validate LLMs

This notebook contains code to validate the performance of large language models (LLMs) on annotations of dimensional emotion scores from the EMOBANK dataset. It compares the LLM annotations to those of trained research staff (see Codebook). It includes checks for annotator transitivity, reliability, and explores at which epsilon values each LLM passes the AltTest.

LLMs examined:
- GPT-5.1
- GPT-5-nano
- Claude-Haiku-4.5
- Claude-Sonnet-4.5

In [ ]:
import sys
import os
import pandas as pd
import pairadigm as pdm
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import re

import dotenv

dotenv.load_dotenv(dotenv_path='/Users/mlchrzan/Library/CloudStorage/OneDrive-Personal/Professional/Data Science/Maryland/bechmark analysis pipeline/benchmark-analysis/.env')

This code pulls together the annotation data from the trained staff. 

In [ ]:
# Initialize dataframes for each concept
valence_df = None
arousal_df = None
dominance_df = None

# Get all annotation files
files = glob.glob('Data/human annotations/Completed/pairs_for_human_annotation - A*.xlsx')

# Process each file
for file in files:
    # Extract annotator number from filename
    match = re.search(r'A(\d+)', file)
    annotator_num = match.group(1)
    
    # Read all sheets
    sheets = pd.read_excel(file, sheet_name=None)
    
    # Process each concept sheet
    for sheet_name, df in sheets.items():
        concept = sheet_name.lower()
        
        # Skip if not a concept sheet
        if concept not in ['valence', 'arousal', 'dominance']:
            continue
        
        # Create decision column name
        decision_col = f'A{annotator_num}'
        
        # Rename concept column and map values
        df_copy = df.copy()
        # Make the fifth column name (the prompt given to the annotators) the name of the concept
        df_copy.rename(columns={df_copy.columns[4]: concept}, inplace=True)
        df_copy[decision_col] = df_copy[concept].map({1: 'Text1', 2: 'Text2', 0: 'Tie'})

        # Adjust the Notes column to be a string and rename it with the annotator_num
        if 'Notes' in df_copy.columns:
            df_copy['Notes'] = df_copy['Notes'].astype(str)
            df_copy['Notes'] = df_copy['Notes'].replace('nan', '')
            df_copy.rename(columns={'Notes': f'Notes_A{annotator_num}'}, inplace=True)
            notes_col = f"Notes_A{annotator_num}"
        
        # Keep only necessary columns
        base_cols = ['item1', 'item2', 'text1', 'text2']
        df_copy = df_copy[base_cols + [decision_col] + [notes_col]]
        
        # Merge with appropriate concept dataframe
        if concept == 'valence':
            if valence_df is None:
                valence_df = df_copy
            else:
                valence_df = valence_df.merge(df_copy, on=base_cols, how='outer')
        elif concept == 'arousal':
            if arousal_df is None:
                arousal_df = df_copy
            else:
                arousal_df = arousal_df.merge(df_copy, on=base_cols, how='outer')
        elif concept == 'dominance':
            if dominance_df is None:
                dominance_df = df_copy
            else:
                dominance_df = dominance_df.merge(df_copy, on=base_cols, how='outer')

print(f"Valence annotations: {valence_df.shape}")
print(f"Arousal annotations: {arousal_df.shape}")
print(f"Dominance annotations: {dominance_df.shape}")

In [ ]:
# Load CGCOT Prompts
with open('Data/cgcot_prompts/valence.txt', 'r') as file:
    val_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/arousal.txt', 'r') as file:
    aro_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/dominance.txt', 'r') as file:
    dom_prompts = file.read().splitlines()

# Prep pairadigms

In [ ]:
# Initialize Pairadigm objects
p_val = pdm.Pairadigm(
    data=valence_df,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in valence_df.columns if col.startswith('A')],
    cgcot_prompts=val_prompts,
    model_name=["gpt-5.1", 
                'gpt-5-nano',
                "claude-sonnet-4-5",
                'claude-haiku-4-5'],
    api_key=[os.getenv('OPENAI_API_KEY'), 
             os.getenv('OPENAI_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY')],
    base_url=["https://us.api.openai.com/v1", 
              "https://us.api.openai.com/v1",
              None, None],
    target_concept='valence'
)

p_aro = pdm.Pairadigm(
    data=arousal_df,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in arousal_df.columns if col.startswith('A')],
    cgcot_prompts=aro_prompts,
    model_name=["gpt-5.1",
                'gpt-5-nano', 
                "claude-sonnet-4-5",
                'claude-haiku-4-5'],
    api_key=[os.getenv('OPENAI_API_KEY'), 
             os.getenv('OPENAI_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY')],
    base_url=["https://us.api.openai.com/v1", 
              "https://us.api.openai.com/v1",
              None, None],
    target_concept='arousal'
)

p_dom = pdm.Pairadigm(
    data=dominance_df,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in dominance_df.columns if col.startswith('A')],
    cgcot_prompts=dom_prompts,
    model_name=["gpt-5.1", 
                'gpt-5-nano',
                "claude-sonnet-4-5",
                'claude-haiku-4-5'],
    api_key=[os.getenv('OPENAI_API_KEY'), 
             os.getenv('OPENAI_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY')],
    base_url=["https://us.api.openai.com/v1", 
              "https://us.api.openai.com/v1",
              None, None],
    target_concept='dominance'
)

In [ ]:
p_dom.get_clients_info()

# Build pairadigms

In [ ]:
for p in [p_val, p_aro, p_dom]:
    p.generate_breakdowns_from_paired()
# run time: 20 minutes

In [ ]:
for p in [p_val, p_aro, p_dom]:
    p.generate_pairwise_annotations(allow_ties=True)
# run time: 30 minutes

# Validation

## IRR

In [ ]:
for p in [p_val, p_aro, p_dom]:
    p.irr()

## Transitivity

In [ ]:
for p in [p_val, p_aro, p_dom]:
    display(p.check_transitivity())

## AltTest

In [ ]:
epsilon_val = p_val.plot_epsilon_sensitivity(return_data=True)
epsilon_val = pd.DataFrame(epsilon_val[1])
epsilon_val['construct'] = 'valence'

In [ ]:
epsilon_aro = p_aro.plot_epsilon_sensitivity(return_data=True)
epsilon_aro = pd.DataFrame(epsilon_aro[1])
epsilon_aro['construct'] = 'arousal'

In [ ]:
epsilon_dom = p_dom.plot_epsilon_sensitivity(return_data=True)
epsilon_dom = pd.DataFrame(epsilon_dom[1])
epsilon_dom['construct'] = 'dominance'

In [ ]:
epsilon_df = pd.concat([epsilon_val, epsilon_aro, epsilon_dom], axis=0)
epsilon_df

In [ ]:
# Get the advantage probability for each model and construct
ad_prob_df = epsilon_df[['advantage_prob', 'model', 'construct']].drop_duplicates()
ad_prob_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define a consistent color palette for models
model_colors = {
    'gpt-5.1': '#1f77b4',
    'gpt-5-nano': '#ff7f0e',
    'claude-sonnet-4-5': '#2ca02c',
    'claude-haiku-4-5': '#d62728'
}

# Create faceted bar plot
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
constructs = ['valence', 'arousal', 'dominance']

for idx, (ax, construct) in enumerate(zip(axes, constructs)):
    # Filter data for this construct
    construct_data = ad_prob_df[ad_prob_df['construct'] == construct].sort_values('advantage_prob', ascending=False)
    
    # Map colors based on model
    colors = [model_colors.get(model, '#808080') for model in construct_data['model']]
    
    # Create bar plot
    bars = ax.bar(construct_data['model'], construct_data['advantage_prob'], 
                   color=colors, 
                   edgecolor='black', linewidth=1.2)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Formatting
    ax.set_title(construct.capitalize(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Model', fontsize=11, fontweight='bold')
    if idx == 0:
        ax.set_ylabel('Advantage Probability', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(ad_prob_df['advantage_prob']) * 1.1)
    ax.tick_params(axis='x', 
                   #rotation=45, 
                   labelsize=9)
    ax.grid(axis='x', alpha=0, linestyle='--')
    

plt.suptitle('LLM Advantage Probability by Construct', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Dawid-Skene (experimental)

In [ ]:
p_val.dawid_skene_annotator_ranking(random_seed=42)

In [ ]:
p_aro.dawid_skene_annotator_ranking(random_seed=42)

In [ ]:
p_dom.dawid_skene_annotator_ranking(random_seed=42)

# Save pairadigms

In [ ]:
today = pd.Timestamp.now().strftime('%m-%d-%y')
p_val.save(f'Results/pairadigms/valence_pairadigm_{today}.pkl')
p_aro.save(f'Results/pairadigms/arousal_pairadigm_{today}.pkl')
p_dom.save(f'Results/pairadigms/dominance_pairadigm_{today}.pkl')

In [ ]:
today = pd.Timestamp.now().strftime('%m-%d-%y')
# Save pairwise results to CSV
for p in [p_val, p_aro, p_dom]:
    paired_results = p.pairwise_df
    paired_results.to_csv(f'Results/paired_results/{p.target_concept}_paired_results_{today}.csv', index=False)

# Valence AltTest Failure Exploration

In [ ]:
# Define human and LLM annotators, then create agreement heatmaps
for p in [p_val, p_aro, p_dom]:
    human_annotators = [col for col in p.pairwise_df.columns if col.startswith('A')]
    llm_annotators = [col for col in p.pairwise_df.columns if col.startswith('decision')]
    all_annotators = human_annotators + llm_annotators
    
    # Initialize a matrix to store agreement percentages
    agreement_matrix = np.zeros((len(all_annotators), len(all_annotators)))
    
    # Calculate agreement percentages
    for i, llm_col in enumerate(all_annotators):
        for j, human_col in enumerate(all_annotators):
            matches = (p.pairwise_df[llm_col] == p.pairwise_df[human_col]).sum()
            total = len(p.pairwise_df)
            agreement_pct = (matches / total) * 100
            agreement_matrix[i, j] = agreement_pct
    
    # Create clean labels by removing "decision_" prefix
    clean_labels = [col.replace('decision_', '') for col in all_annotators]
    
    # Create a DataFrame for better labeling
    agreement_df = pd.DataFrame(
        agreement_matrix,
        index=clean_labels,
        columns=clean_labels
    )
    
    # Create a mask for the upper triangle
    mask = np.triu(np.ones_like(agreement_df, dtype=bool), k=1)
    
    # Plot the heatmap
    plt.figure(figsize=(12, 10))
    ax = sns.heatmap(
        agreement_df,
        annot=True,
        fmt='.1f',
        cmap='coolwarm',
        vmin=0,
        vmax=100,
        cbar_kws={'label': 'Agreement Percentage (%)'},
        linewidths=0.5,
        linecolor='gray',
        mask=mask
    )
    
    # Add a bold dashed line to separate humans from LLMs
    num_humans = len(human_annotators)
    ax.axhline(y=num_humans, color='black', linewidth=3, linestyle='--')
    ax.axvline(x=num_humans, color='black', linewidth=3, linestyle='--')
    
    # Add italicized text label
    ax.text(len(clean_labels) * 0.95, num_humans + 0.25, 'Human/LLM Boundary', 
            fontsize=12, style='italic', color='gray', ha='right', va='bottom')
    
    plt.title(f'%Agreement Between LLM and Human Annotators ({p.target_concept.capitalize()})', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Define human and LLM annotators, then create agreement heatmaps in one figure
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, p in enumerate([p_val, p_aro, p_dom]):
    human_annotators = [col for col in p.pairwise_df.columns if col.startswith('A')]
    llm_annotators = [col for col in p.pairwise_df.columns if col.startswith('decision')]
    all_annotators = human_annotators + llm_annotators
    
    # Initialize a matrix to store agreement percentages
    agreement_matrix = np.zeros((len(all_annotators), len(all_annotators)))
    
    # Calculate agreement percentages
    for i, llm_col in enumerate(all_annotators):
        for j, human_col in enumerate(all_annotators):
            matches = (p.pairwise_df[llm_col] == p.pairwise_df[human_col]).sum()
            total = len(p.pairwise_df)
            agreement_pct = (matches / total) * 100
            agreement_matrix[i, j] = agreement_pct
    
    # Create clean labels by removing "decision_" prefix
    clean_labels = [col.replace('decision_', '') for col in all_annotators]
    
    # Create a DataFrame for better labeling
    agreement_df = pd.DataFrame(
        agreement_matrix,
        index=clean_labels,
        columns=clean_labels
    )
    
    # Create a mask for the upper triangle
    mask = np.triu(np.ones_like(agreement_df, dtype=bool), k=1)
    
    # Plot the heatmap on the current subplot
    ax = sns.heatmap(
        agreement_df,
        annot=True,
        fmt='.1f',
        cmap='coolwarm',
        vmin=0,
        vmax=100,
        cbar_kws={'label': 'Agreement (%)'},
        linewidths=0.5,
        linecolor='gray',
        mask=mask,
        ax=axes[idx]
    )
    
    # Add a bold dashed line to separate humans from LLMs
    num_humans = len(human_annotators)
    ax.axhline(y=num_humans, color='black', linewidth=2, linestyle='--')
    ax.axvline(x=num_humans, color='black', linewidth=2, linestyle='--')
    
    # Set title for this subplot
    ax.set_title(f'{p.target_concept.capitalize()}', fontsize=12, fontweight='bold')

plt.suptitle('Agreement Between LLM and Human Annotators', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Unpaired AltTest

In [ ]:
# Filter out rows where any column starting with 'A' has the value 'Tie'
valence_df_filtered = valence_df[~valence_df.filter(regex='^A').eq('Tie').any(axis=1)].reset_index(drop=True)
print(f"Number of unique items in valence after filtering ties: {len(pd.unique(valence_df_filtered[['item1', 'item2']].values.ravel()))}")
valence_df_filtered

In [ ]:
# Filter out rows where any column starting with 'A' has the value 'Tie'
arousal_df_filtered = arousal_df[~arousal_df.filter(regex='^A').eq('Tie').any(axis=1)].reset_index(drop=True)
print(f"Number of unique items in arousal after filtering ties: {len(pd.unique(arousal_df_filtered[['item1', 'item2']].values.ravel()))}")
arousal_df_filtered

In [ ]:
# Filter out rows where any column starting with 'A' has the value 'Tie'
dominance_df_filtered = dominance_df[~dominance_df.filter(regex='^A').eq('Tie').any(axis=1)].reset_index(drop=True)
print(f"Number of unique items in dominance after filtering ties: {len(pd.unique(dominance_df_filtered[['item1', 'item2']].values.ravel()))}")
dominance_df_filtered

## Build Unpaired Pairadigms

In [ ]:
# Initialize Pairadigm objects
p_val_unpair = pdm.Pairadigm(
    data=valence_df_filtered,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in valence_df_filtered.columns if col.startswith('A')],
    cgcot_prompts=val_prompts,
    model_name=["gpt-5.1", 
                'gpt-5-nano',
                "claude-sonnet-4-5",
                'claude-haiku-4-5'],
    api_key=[os.getenv('OPENAI_API_KEY'), 
             os.getenv('OPENAI_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY')],
    base_url=["https://us.api.openai.com/v1", 
              "https://us.api.openai.com/v1",
              None, None],
    target_concept='valence'
)

p_aro_unpair = pdm.Pairadigm(
    data=arousal_df_filtered,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in arousal_df_filtered.columns if col.startswith('A')],
    cgcot_prompts=aro_prompts,
    model_name=["gpt-5.1",
                'gpt-5-nano', 
                "claude-sonnet-4-5",
                'claude-haiku-4-5'],
    api_key=[os.getenv('OPENAI_API_KEY'), 
             os.getenv('OPENAI_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY')],
    base_url=["https://us.api.openai.com/v1", 
              "https://us.api.openai.com/v1",
              None, None],
    target_concept='arousal'
)

p_dom_unpair = pdm.Pairadigm(
    data=dominance_df_filtered,
    item_id_cols=['item1', 'item2'],
    item_text_cols=['text1', 'text2'],
    paired=True,
    annotated=True,
    annotator_cols=[col for col in dominance_df_filtered.columns if col.startswith('A')],
    cgcot_prompts=dom_prompts,
    model_name=["gpt-5.1", 
                'gpt-5-nano',
                "claude-sonnet-4-5",
                'claude-haiku-4-5'],
    api_key=[os.getenv('OPENAI_API_KEY'), 
             os.getenv('OPENAI_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY'),
             os.getenv('ANTHROPIC_API_KEY')],
    base_url=["https://us.api.openai.com/v1", 
              "https://us.api.openai.com/v1",
              None, None],
    target_concept='dominance'
)

In [ ]:
for p in [p_val_unpair, p_aro_unpair, p_dom_unpair]:
    p.generate_breakdowns_from_paired()
# run time: 20 minutes

In [ ]:
for p in [p_val_unpair, p_aro_unpair, p_dom_unpair]:
    p.generate_pairwise_annotations(allow_ties=True)
# run time: 22 minutes

In [ ]:
today = pd.Timestamp.now().strftime('%m-%d-%y')

p_val_unpair.save(f'Results/pairadigms/untied_valence_pairadigm_{today}.pkl')
p_aro_unpair.save(f'Results/pairadigms/untied_arousal_pairadigm_{today}.pkl')
p_dom_unpair.save(f'Results/pairadigms/untied_dominance_pairadigm_{today}.pkl')

# Save pairwise results to CSV
for p in [p_val_unpair, p_aro_unpair, p_dom_unpair]:
    paired_results = p.pairwise_df
    paired_results.to_csv(f'Results/paired_results/untied_{p.target_concept}_paired_results_{today}.csv', index=False)

## Validation

In [ ]:
epsilon_val_untied = p_val_unpair.plot_epsilon_sensitivity(return_data=True)
epsilon_val_untied = pd.DataFrame(epsilon_val_untied[1])
epsilon_val_untied['construct'] = 'valence'

In [ ]:
epsilon_aro_untied = p_aro_unpair.plot_epsilon_sensitivity(return_data=True)
epsilon_aro_untied = pd.DataFrame(epsilon_aro_untied[1])
epsilon_aro_untied['construct'] = 'arousal'

In [ ]:
epsilon_dom_untied = p_dom_unpair.plot_epsilon_sensitivity(return_data=True)
epsilon_dom_untied = pd.DataFrame(epsilon_dom_untied[1])
epsilon_dom_untied['construct'] = 'dominance'

In [ ]:
epsilon_df_untied = pd.concat([epsilon_val_untied, epsilon_aro_untied, epsilon_dom_untied], axis=0)
epsilon_df_untied

In [ ]:
# Get the advantage probability for each model and construct
ad_prob_df_untied = epsilon_df_untied[['advantage_prob', 'model', 'construct']].drop_duplicates()
ad_prob_df_untied

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define a consistent color palette for models
model_colors = {
    'gpt-5.1': '#1f77b4',
    'gpt-5-nano': '#ff7f0e',
    'claude-sonnet-4-5': '#2ca02c',
    'claude-haiku-4-5': '#d62728'
}

# Create faceted bar plot
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
constructs = ['valence', 'arousal', 'dominance']

for idx, (ax, construct) in enumerate(zip(axes, constructs)):
    # Filter data for this construct
    construct_data = ad_prob_df_untied[ad_prob_df_untied['construct'] == construct].sort_values('advantage_prob', ascending=False)
    
    # Map colors based on model
    colors = [model_colors.get(model, '#808080') for model in construct_data['model']]
    
    # Create bar plot
    bars = ax.bar(construct_data['model'], construct_data['advantage_prob'], 
                   color=colors, 
                   edgecolor='black', linewidth=1.2)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Formatting
    ax.set_title(construct.capitalize(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Model', fontsize=11, fontweight='bold')
    if idx == 0:
        ax.set_ylabel('Advantage Probability', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(ad_prob_df_untied['advantage_prob']) * 1.1)
    ax.tick_params(axis='x', 
                   #rotation=45, 
                   labelsize=9)
    ax.grid(axis='x', alpha=0, linestyle='--')
    

plt.suptitle('LLM Advantage Probability by Construct', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Requires pairadigm v0.5.1
p_val_unpair = pdm.Pairadigm.load(f'Results/pairadigms/untied_valence_pairadigm_12-19-25.pkl')
p_aro_unpair = pdm.Pairadigm.load(f'Results/pairadigms/untied_arousal_pairadigm_12-19-25.pkl')
p_dom_unpair = pdm.Pairadigm.load(f'Results/pairadigms/untied_dominance_pairadigm_12-19-25.pkl')

In [ ]:
val_DS = p_val_unpair.dawid_skene_annotator_ranking(random_seed=42)
aro_DS = p_aro_unpair.dawid_skene_annotator_ranking(random_seed=42)
dom_DS = p_dom_unpair.dawid_skene_annotator_ranking(random_seed=42)

In [ ]:
val_DS['construct'] = 'valence'
aro_DS['construct'] = 'arousal'
dom_DS['construct'] = 'dominance'

dawid_skene_df = pd.concat([val_DS, aro_DS, dom_DS], axis=0)
dawid_skene_df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Defensive copy + typing ---
df = dawid_skene_df.copy()
df["reliability"] = pd.to_numeric(df["reliability"], errors="coerce")
df["rank"] = pd.to_numeric(df["rank"], errors="coerce").astype("Int64")

# Optional label cleanup for display
label_map = {
    "decision_gpt-5.1": "GPT-5.1",
    "decision_gpt-5-nano": "GPT-5-nano",
    "decision_claude-sonnet-4-5": "Claude-Sonnet-4.5",
    "decision_claude-haiku-4-5": "Claude-Haiku-4.5",
}
df["annotator_label"] = df["annotator"].map(label_map).fillna(df["annotator"])

construct_order = ["valence", "arousal", "dominance"]
df["construct"] = pd.Categorical(df["construct"], categories=construct_order, ordered=True)

# --- Publication-style theme (larger fonts) ---
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 600,
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

type_colors = {"Human": "#4C78A8", "LLM": "#F58518"}

xmin = max(0, df["reliability"].min() - 0.06)
xmax_data = df["reliability"].max()
xmax = min(1.08, xmax_data + 0.08)  # extra room for value labels
x_offset = (xmax - xmin) * 0.02      # larger text offset to avoid marker overlap

fig, axes = plt.subplots(1, 3, figsize=(16, 6.5), sharex=True, sharey=False)

for ax, construct in zip(axes, construct_order):
    sub = (
        df[df["construct"] == construct]
        .sort_values("rank", ascending=True)
        .reset_index(drop=True)
    )

    y = np.arange(len(sub))
    ax.hlines(y=y, xmin=xmin, xmax=sub["reliability"], color="#D3D3D3", lw=2.2, zorder=1)

    for t, d in sub.groupby("type", sort=False):
        ax.scatter(
            d["reliability"], d.index.values,
            s=120, color=type_colors.get(t, "gray"),
            edgecolor="black", linewidth=0.7, zorder=3, label=t
        )

    ylabels = [f"{r}. {a}" for r, a in zip(sub["rank"], sub["annotator_label"])]
    ax.set_yticks(y)
    ax.set_yticklabels(ylabels)
    ax.invert_yaxis()

    # Value labels: fixed leading zero + white bbox so marker/line never hides text
    for i, val in enumerate(sub["reliability"]):
        ax.text(
            val + x_offset, i, f"{val:0.3f}",
            va="center", ha="left", fontsize=11, zorder=5, clip_on=False,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.9, pad=0.6)
        )

    ax.set_title(construct.capitalize(), pad=12, weight="bold")
    ax.set_xlim(xmin, xmax)
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    ax.grid(axis="y", alpha=0)

axes[0].set_ylabel("Annotator (ranked)")
for ax in axes:
    ax.set_xlabel("Dawid–Skene Reliability")

handles, labels = axes[0].get_legend_handles_labels()
uniq = dict(zip(labels, handles))
fig.legend(
    uniq.values(), uniq.keys(), title="Annotator Type",
    loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.03)
)

fig.suptitle("Dawid–Skene Annotator Reliability by Construct", y=1.08, weight="bold", fontsize=18)
fig.tight_layout(rect=[0, 0, 1, 0.98])

out_path = "Results/dawid_skene_reliability_by_construct.png"
plt.savefig(out_path, bbox_inches="tight")
plt.show()

print(f"Saved: {out_path}")